# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:
# Load the dataset
df = pd.read_csv('AviationData.csv', low_memory=False, encoding='latin-1')

# Parse event date
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())


In [ ]:
# Preview first rows
df.head()

In [ ]:
# Inspect data types
print(df.dtypes)


In [ ]:
# Inspect NaN counts
nan_counts = df.isnull().sum()
print("NaN counts per column:")
print(nan_counts)


In [ ]:
# Summary statistics
df.describe()


## Data Cleaning

### Filtering aircrafts and events

In [ ]:
# Inspect relevant filtering columns
print("Amateur.Built values:")
print(df['Amateur.Built'].value_counts(dropna=False))
print("\nAircraft.Category values:")
print(df['Aircraft.Category'].value_counts(dropna=False))
print("\nEvent.Date year range:", df['Event.Date'].dt.year.min(), "-", df['Event.Date'].dt.year.max())


In [ ]:
# --- Filtering ---

# 1. Keep only professional (non-amateur) builds
df = df[df['Amateur.Built'] == 'No']

# 2. Keep only fixed-wing Airplanes (client is focused on commercial/passenger airplanes, not helicopters, gliders, etc.)
df = df[df['Aircraft.Category'] == 'Airplane']

# 3. Filter to 1983 onwards (40-year max active lifetime from 2023)
df = df[df['Event.Date'].dt.year >= 1983]

print("Shape after filtering:", df.shape)


We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [ ]:
# Inspect injury columns
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 
               'Total.Minor.Injuries', 'Total.Uninjured']
print("NaN counts in injury columns:")
print(df[injury_cols].isnull().sum())
print("\nSample:")
print(df[injury_cols].head(10))


**Imputation assumption:** Missing injury values are filled with 0. 
If no fatalities/serious injuries were reported, it is reasonable to assume there were none 
(reporters would typically note fatalities). This is a conservative assumption.

We estimate total onboard passengers as the sum of all injury category counts.
Rows where total onboard is zero are excluded from injury rate calculation (undefined denominator).

**Injury Rate** = (Total Fatal + Total Serious Injuries) / Total Onboard
This gives a per-passenger probability of serious harm in the event of an accident.


In [ ]:
# Impute NaN injury values with 0
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 
               'Total.Minor.Injuries', 'Total.Uninjured']
df[injury_cols] = df[injury_cols].fillna(0)

# Estimate total passengers onboard
df['Total.Onboard'] = df[injury_cols].sum(axis=1)

# Create fatal/serious injury count
df['Fatal_Serious_Count'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']

# Compute injury rate: fraction of passengers fatally or seriously injured
# Rows with zero onboard are excluded (NaN denominator)
df['Injury_Rate'] = df['Fatal_Serious_Count'] / df['Total.Onboard'].replace(0, np.nan)

print("Injury_Rate summary statistics:")
print(df['Injury_Rate'].describe())
print(f"\nRows with defined injury rate: {df['Injury_Rate'].notna().sum()}")


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

In [ ]:
# Inspect Aircraft.damage column
print("Aircraft.damage value counts:")
print(df['Aircraft.damage'].value_counts(dropna=False))


In [ ]:
# Standardize Aircraft.damage (strip whitespace)
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip()

# Create binary 'Destroyed' column: 1 if aircraft was destroyed, 0 otherwise
df['Destroyed'] = (df['Aircraft.damage'].str.lower() == 'destroyed').astype(int)

print("Destroyed column distribution:")
print(df['Destroyed'].value_counts())
print(f"\nDestruction rate: {df['Destroyed'].mean():.3f} ({df['Destroyed'].mean()*100:.1f}%)")


**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

**Make column cleaning tasks:**
1. Strip leading/trailing whitespace
2. Standardize casing (title case)
3. Consolidate known variant spellings of the same manufacturer (e.g. 'Cirrus Design Corp', 'Cirrus Design Corp.', 'Cirrus Design Corporation' → 'Cirrus')
4. Drop rows with missing Make
5. Keep only Makes with ≥ 50 accident records for statistical robustness


In [ ]:
# Inspect Make column
print("Sample raw Make values:")
print(df['Make'].value_counts().head(30))
print(f"\nTotal unique Make values: {df['Make'].nunique()}")


In [ ]:
# Clean Make column
df['Make'] = df['Make'].str.strip().str.title()

# Consolidate common variant spellings for the same manufacturer
make_map = {
    'Cirrus Design Corp.':        'Cirrus',
    'Cirrus Design Corp':         'Cirrus',
    'Cirrus Design Corporation':  'Cirrus',
    'Cirrus Design':              'Cirrus',
    'Mcdonnell Douglas Aircraft Co':    'Mcdonnell Douglas',
    'Mcdonnell Douglas Corporation':    'Mcdonnell Douglas',
    'Mcdonnell-Douglas':                'Mcdonnell Douglas',
    'Air Tractor Inc':            'Air Tractor',
    'Air Tractor Inc.':           'Air Tractor',
    'Boeing Commercial Airplane': 'Boeing',
    'Cessna Aircraft':            'Cessna',
    'Cessna Aircraft Company':    'Cessna',
    'Cessna Aircraft Co':         'Cessna',
    'Piper Aircraft':             'Piper',
    'Piper Aircraft Inc':         'Piper',
    'Beechcraft':                 'Beech',
    'Beech Aircraft':             'Beech',
    'Beech Aircraft Corp':        'Beech',
}
df['Make'] = df['Make'].replace(make_map)

# Drop rows with missing Make
df = df.dropna(subset=['Make'])

print("After cleaning - Top Make counts:")
print(df['Make'].value_counts().head(20))
print(f"\nTotal unique Makes: {df['Make'].nunique()}")


In [ ]:
# Keep only Makes with >= 50 incident records (statistical robustness threshold)
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]

print(f"Makes with >= 50 incidents: {len(valid_makes)}")
print(f"Dataframe shape after filtering: {df.shape}")


In [ ]:
# Inspect Model column
print(f"Model NaN count: {df['Model'].isnull().sum()}")
print("\nSample Model values:")
print(df[['Make','Model']].dropna().head(20))


In [ ]:
# Check if model names are unique to each Make
model_make_count = df.groupby('Model')['Make'].nunique()
shared_models = model_make_count[model_make_count > 1]
print(f"Model names shared across multiple Makes: {len(shared_models)}")
print("\nExamples:")
print(shared_models.head(10))


In [ ]:
# Model labels are NOT unique across makes (e.g. '172' appears for both Cessna and others)
# Drop rows with missing Model
df = df.dropna(subset=['Model'])

# Create a unique Make_Model identifier combining Make and Model
df['Make_Model'] = df['Make'] + ' ' + df['Model']

print(f"Unique Make_Model combinations: {df['Make_Model'].nunique()}")
print("\nSample Make_Model values:")
print(df['Make_Model'].value_counts().head(10))


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
# --- Engine.Type ---
print("Engine.Type value counts:")
print(df['Engine.Type'].value_counts(dropna=False))


In [ ]:
# Clean Engine.Type: strip, title case, map unknowns to NaN
df['Engine.Type'] = df['Engine.Type'].str.strip().str.title()
df['Engine.Type'] = df['Engine.Type'].replace({'Unknown': np.nan, 'Unk': np.nan, 'None': np.nan})

print("Engine.Type after cleaning:")
print(df['Engine.Type'].value_counts(dropna=False))


In [ ]:
# --- Weather.Condition ---
print("Weather.Condition value counts:")
print(df['Weather.Condition'].value_counts(dropna=False))


In [ ]:
# Clean Weather.Condition: standardize and map unknowns to NaN
df['Weather.Condition'] = df['Weather.Condition'].str.strip().str.upper()
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': np.nan, 'UNKN': np.nan})

print("Weather.Condition after cleaning:")
print(df['Weather.Condition'].value_counts(dropna=False))


In [ ]:
# --- Number.of.Engines ---
print("Number.of.Engines value counts:")
print(df['Number.of.Engines'].value_counts(dropna=False))


In [ ]:
# Convert to numeric; invalid entries become NaN
df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')

print("Number.of.Engines after cleaning:")
print(df['Number.of.Engines'].value_counts(dropna=False))


In [ ]:
# --- Purpose.of.flight ---
print("Purpose.of.flight value counts:")
print(df['Purpose.of.flight'].value_counts(dropna=False))


In [ ]:
# Standardize: title case, map 'Unknown' to NaN
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip().str.title()
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({'Unknown': np.nan})

print("Purpose.of.flight after cleaning:")
print(df['Purpose.of.flight'].value_counts(dropna=False).head(15))


In [ ]:
# --- Broad.phase.of.flight ---
print("Broad.phase.of.flight value counts:")
print(df['Broad.phase.of.flight'].value_counts(dropna=False))


In [ ]:
# Standardize: title case, map 'Unknown' to NaN
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip().str.title()
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace({'Unknown': np.nan})

print("Broad.phase.of.flight after cleaning:")
print(df['Broad.phase.of.flight'].value_counts(dropna=False))


In [ ]:
# Compute NaN fraction for all columns
nan_fractions = df.isnull().mean().sort_values(ascending=False)
print("NaN fraction per column:")
print(nan_fractions)


In [ ]:
# Drop columns with > 50% missing values
threshold = 0.5
cols_to_drop = nan_fractions[nan_fractions > threshold].index.tolist()
print(f"Columns to drop (>{threshold*100}% NaN): {cols_to_drop}")

df = df.drop(columns=cols_to_drop)
print(f"\nFinal dataframe shape: {df.shape}")
print("Remaining columns:", df.columns.tolist())


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [ ]:
# Save cleaned dataframe to CSV for use in analysis notebook
df.to_csv('aviation_cleaned.csv', index=False)
print("Saved cleaned data to aviation_cleaned.csv")
print(f"Final dataset shape: {df.shape}")
df.head()
